# 01 · Clasificación binaria: de regresión logística a decisiones

**Laboratorio ML — scikit-learn**

**Objetivos**
1. Comparar una regresión logística contra un baseline.
2. Examinar una matriz de confusión, *precision*, *recall*, F1, ROC-AUC y Average Precision.
3. Elegir un umbral de decisión con **validación** y estimar el resultado final con **test**.
4. Explorar una función de costo en la que los falsos negativos son más caros que las falsas alarmas.

**Escenario:** detección de fallas en una máquina. Clase **0 = normal**, clase **1 = falla**.

> Los datos son **sintéticos**, generados para experimentar con clasificación desbalanceada. Las 8 características **no corresponden** a sensores físicos concretos.

**Cómo usarlo:** ejecuta las celdas de arriba abajo. Después modifica los parámetros señalados en «Para experimentar» y vuelve a ejecutar. Funciona en Jupyter o Google Colab con NumPy, pandas, Matplotlib y scikit-learn.

## 1. Generación y separación de datos

Produciremos una clase positiva minoritaria (aproximadamente 5 %). Reservaremos **20 % para test** antes de cualquier selección de modelo o umbral, y dividiremos el resto en entrenamiento y validación.

La estratificación preserva aproximadamente la proporción de clases en los tres conjuntos.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42

X, y = make_classification(
    n_samples=2500,
    n_features=8,
    n_informative=4,
    n_redundant=2,
    weights=[0.95, 0.05],
    class_sep=1.5,
    flip_y=0.01,
    random_state=RANDOM_STATE,
)

X_dev, X_test, y_dev, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)
X_train, X_val, y_train, y_val = train_test_split(
    X_dev, y_dev, test_size=0.25,
    stratify=y_dev, random_state=RANDOM_STATE
)

for name, labels in [("Train", y_train), ("Validation", y_val), ("Test", y_test)]:
    count = np.bincount(labels, minlength=2)
    print(f"{name:10s} total={len(labels):4d} | normales={count[0]:4d} | fallas={count[1]:3d} | prevalencia={labels.mean():.1%}")

## 2. Baseline: ¿cuánto vale una accuracy elevada?

Un clasificador que predice siempre la clase más frecuente puede parecer «bueno» si solo observamos accuracy, pero será inútil para encontrar fallas.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, recall_score

baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)

y_base = baseline.predict(X_val)

print(f"Accuracy baseline: {accuracy_score(y_val, y_base):.3f}")
print(f"Recall baseline:   {recall_score(y_val, y_base, zero_division=0):.3f}")

**Pregunta:** ¿por qué el baseline tiene accuracy cercana al 94 % y recall igual a cero? Relaciona el resultado con la prevalencia de fallas.

## 3. Regresión logística y prevención de data leakage

El pipeline garantiza que la estandarización se ajuste **solo con train**:

\[
\hat p = \sigma(\mathbf{w}^{T}\mathbf{x}+b),\qquad
\sigma(z)=\frac{1}{1+e^{-z}}
\]



In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
)
model.fit(X_train, y_train)

positive_index = list(model.classes_).index(1)
p_val = model.predict_proba(X_val)[:, positive_index]

print("Probabilidades mín/máx:", p_val.min(), p_val.max())

## 4. Matriz de confusión y métricas para distintos umbrales

La clase positiva es **falla = 1**. Las métricas que usaremos son:

\[
\mathrm{Precision}=\frac{TP}{TP+FP},\qquad
\mathrm{Recall}=\frac{TP}{TP+FN}
\]

\[
F_1=2\frac{\mathrm{Precision}\,\mathrm{Recall}}{\mathrm{Precision}+\mathrm{Recall}}
\]

El umbral transforma una probabilidad estimada en una decisión. **No modifica los coeficientes aprendidos.**

In [ ]:
from sklearn.metrics import (
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
)

def evaluate_threshold(y_true, probabilities, threshold):
    predicted = (probabilities >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(
        y_true, predicted, labels=[0, 1]
    ).ravel()
    return {
        "threshold": threshold,
        "accuracy": accuracy_score(y_true, predicted),
        "precision": precision_score(y_true, predicted, zero_division=0),
        "recall": recall_score(y_true, predicted, zero_division=0),
        "f1": f1_score(y_true, predicted, zero_division=0),
        "TP": int(tp), "FP": int(fp), "FN": int(fn), "TN": int(tn),
    }

comparison = pd.DataFrame([
    evaluate_threshold(y_val, p_val, t)
    for t in [0.10, 0.30, 0.50]
])
display(comparison.round(3))

**Para experimentar:** añade umbrales 0.05 y 0.70. ¿Qué ocurre con FP, FN, precision y recall? ¿Por qué reducir el umbral no garantiza mejorar precision?

## 5. Curvas ROC y Precision–Recall

**ROC** contrasta *true positive rate* frente a *false positive rate*. **PR** contrasta precision frente a recall al variar el umbral.

- **ROC-AUC** resume la capacidad de ordenar positivos por encima de negativos.
- **Average Precision (AP)** resume precision ponderada por incrementos de recall. No es necesariamente igual al área trapezoidal bajo PR.
- Para una clase positiva rara, la curva PR permite apreciar cuántas alarmas son realmente útiles.

Estas curvas se calculan sobre **puntuaciones/probabilidades**, no sobre las etiquetas ya umbralizadas.

In [ ]:
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    RocCurveDisplay,
    PrecisionRecallDisplay,
)

print(f"ROC-AUC (validation): {roc_auc_score(y_val, p_val):.3f}")
print(f"AP      (validation): {average_precision_score(y_val, p_val):.3f}")

RocCurveDisplay.from_predictions(y_val, p_val)
plt.title("ROC · validación")
plt.show()

PrecisionRecallDisplay.from_predictions(y_val, p_val)
plt.title("Precision–Recall · validación")
plt.show()

print(f"Prevalencia clase positiva: {y_val.mean():.2%}")

**Pregunta:** compara Average Precision con la prevalencia de la clase positiva. ¿Por qué un ROC-AUC razonablemente alto no garantiza que la mayoría de las alarmas sean verdaderas?

## 6. Selección de umbral mediante F1

La elección se realiza **con datos de validación**, no usando test. Evaluaremos una rejilla discreta de umbrales y elegiremos el que maximiza F1. Esta selección es didáctica: un objetivo operativo real podría requerir otro criterio.

In [ ]:
thresholds = np.linspace(0.05, 0.95, 19)
f1_values = np.array([
    f1_score(y_val, p_val >= t, zero_division=0)
    for t in thresholds
])

best_index = int(np.argmax(f1_values))
threshold_f1 = float(thresholds[best_index])

print(f"Umbral seleccionado por F1: {threshold_f1:.2f}")
print(f"F1 en validación:          {f1_values[best_index]:.3f}")

plt.figure(figsize=(7, 4))
plt.plot(thresholds, f1_values, marker="o")
plt.axvline(threshold_f1, linestyle="--", label=f"F1 óptimo: {threshold_f1:.2f}")
plt.xlabel("Threshold")
plt.ylabel("F1")
plt.title("F1 vs. threshold · validación")
plt.legend()
plt.grid(alpha=0.2)
plt.show()

## 7. Función de costo: una decisión distinta

Supongamos (solo para el ejercicio) estos costos constantes:

- **Falso negativo (FN):** 10,000 unidades monetarias.
- **Falso positivo (FP):** 500 unidades monetarias.

\[
J(t)=10\,000\,FN(t)+500\,FP(t)
\]

Minimizar $J$ puede seleccionar un umbral diferente del que maximiza F1. En aplicaciones reales, esos costos deben fundamentarse con expertos del dominio; la frecuencia y gravedad de distintos tipos de falla también pueden variar.

In [ ]:
COST_FN = 10_000
COST_FP = 500

def operational_cost(metrics):
    return COST_FN * metrics["FN"] + COST_FP * metrics["FP"]

cost_values = np.array([
    operational_cost(evaluate_threshold(y_val, p_val, float(t)))
    for t in thresholds
])
cost_index = int(np.argmin(cost_values))
threshold_cost = float(thresholds[cost_index])

print(f"Umbral por F1:    {threshold_f1:.2f}")
print(f"Umbral por costo: {threshold_cost:.2f}")
print(f"Costo mínimo estimado en validación: {cost_values[cost_index]:,.0f}")

plt.figure(figsize=(7, 4))
plt.plot(thresholds, cost_values, marker="o")
plt.axvline(threshold_cost, linestyle="--", label=f"Costo mínimo: {threshold_cost:.2f}")
plt.xlabel("Threshold")
plt.ylabel("Costo en conjunto de validación")
plt.title("Costo operativo vs. threshold")
plt.legend()
plt.grid(alpha=0.2)
plt.show()

**Para experimentar:** cambia \`COST_FN\` de 10,000 a 1,000 y después a 50,000. ¿Cómo cambia el umbral seleccionado? ¿Coincide con el mejor umbral según F1?

> **Nota:** como el conjunto de validación solo contiene unas decenas de fallas, los umbrales seleccionados pueden variar bastante con una partición distinta. En un proyecto serio convendría verificar la estabilidad con más datos o validación cruzada.

## 8. Evaluación final (una sola vez) en test

**Ejecuta esta celda después de fijar** el modelo y los dos umbrales elegidos con validación. No uses los resultados de test para seguir ajustando decisiones.

Aquí comparamos, a modo de informe final, los dos criterios seleccionados antes de mirar test.

In [ ]:
p_test = model.predict_proba(X_test)[:, positive_index]

test_results = pd.DataFrame([
    {
        "criterio": "F1 en validación",
        **evaluate_threshold(y_test, p_test, threshold_f1),
    },
    {
        "criterio": "Costo en validación",
        **evaluate_threshold(y_test, p_test, threshold_cost),
    },
])
test_results["costo_realizado"] = [
    operational_cost(row)
    for row in test_results.to_dict("records")
]
display(test_results.round(3))

print(f"ROC-AUC en test: {roc_auc_score(y_test, p_test):.3f}")
print(f"AP en test:      {average_precision_score(y_test, p_test):.3f}")

## 9. Conclusiones y retos

1. ¿Por qué es engañosa accuracy cuando la clase positiva es rara?
2. ¿Qué cambia al reducir el umbral? ¿Y qué permanece igual?
3. ¿Cuál de los dos criterios de selección de umbral te parece más adecuado para mantenimiento industrial?
4. ¿Por qué debemos elegir el umbral usando validación y no test?
5. Repite el notebook con \`weights=[0.90, 0.10]\`, \`class_sep=0.8\` o \`flip_y=0.05\`. **Vuelve a comenzar desde la generación de datos** y compara los resultados.
6. Como extensión, sustituye la validación simple por **StratifiedKFold** para explorar la estabilidad del umbral.

**Notas conceptuales relacionadas en Obsidian:** [[Regresión logística]], [[Métricas de clasificación]], [[Umbral de decisión]], [[Curvas ROC y Precision-Recall]], [[Train-validation-test]], [[Cross-validation]].

Este notebook sirve como laboratorio de aprendizaje; para un proyecto productivo faltarían análisis de calibración, deriva de datos, costos sustentados y validación adaptada al esquema temporal o por equipos, cuando corresponda.